# 07-units-T (Medellin, O3) -- dimensionally constrained search with temperature in the pool

Same set-up as `bogota/07_o3_units_with_temp_equation_discovery.ipynb`: the existing same-day no-lag selection plus `temp_2m` (unit `K`, a temperature difference), every variable with a unit, PySR restricted to dimensionally homogeneous expressions. Ozone in Medellin is reported in ug m-3 (`kg/m^3`); `pmc` = PM10 - PM2.5. Three seeds.

Scoring (same protocol as the Bogota lead equations, notebooks A2/A4): leave-one-year-out, constants refitted by non-linear least squares on the training years, **both** pooled R2 and mean of per-year R2, on the rows the search used. The existing free-form front is scored on the same rows.

In [1]:
# %% Settings (edit here, then Run All)
CITY, TARGET, TAG = 'medellin', 'o3', 'units_temp'
MASTER = 'medellin_o3_master_dataset.parquet'                         # file in Data/processed/colombia/<city>/
SELECTION_JSON = 'medellin_o3_feature_selection.json'                    # its selected_no_lag is the pool (identical to the free-form run)
EXTRA_FEATURES = ['temp_2m']                  # added to that pool (temperature offered, as for Bogota)
DROP_FEATURES = []                    # removed from it (split into raw factors)
SEEDS = [0, 1, 2]                           # symbolic search is stochastic: report all of them
PYSR_KW = dict(niterations=200, binary_operators=["+", "-", "*", "/"], unary_operators=["log", "sqrt", "exp"],
               model_selection="best", maxsize=25, populations=24)     # identical to the Bogota constrained runs
UNIT_MAP = {'radiation_station': 'W/m^2', 'pm25_lag1': 'kg/m^3', 'wind_speed': 'm/s', 'no_sisaire': 'kg/m^3', 'no2': 'kg/m^3', 'pmc_lag2': 'kg/m^3', 'temp_2m': 'K'}
Y_UNITS = 'kg/m^3'
FREEFORM_FRONT = 'medellin_o3_pysr_pareto_front_refresh.csv'                     # the existing free-form front, scored here on the same rows
MAX_COMPLEXITY_REPORTED = 25

In [2]:
import sys, json, re, warnings
from pathlib import Path
PROJECT_ROOT = Path("/path/to/project")
sys.path.insert(0, str(PROJECT_ROOT / "Scripts"))
import numpy as np, pandas as pd, sympy as sp
from scipy.optimize import curve_fit
from sklearn.metrics import r2_score
DATA_DIR = PROJECT_ROOT / "Data" / "processed" / "colombia" / CITY
df = pd.read_parquet(DATA_DIR / MASTER)
df["date"] = pd.to_datetime(df["date"]); df["year"] = df["date"].dt.year
df = df.dropna(subset=[TARGET]).reset_index(drop=True)
selection = json.load(open(DATA_DIR / SELECTION_JSON))
features = [f for f in selection["selected_no_lag"] if f not in DROP_FEATURES]
for f in EXTRA_FEATURES:
    if f not in features: features.append(f)
print("pool:", features)
data = df.dropna(subset=features + [TARGET]).reset_index(drop=True)
X = data[features].to_numpy(); y = data[TARGET].to_numpy()
X_units = [UNIT_MAP[f] for f in features]
print(data.shape, dict(zip(features, X_units)), "years:", sorted(data.year.unique()))

pool: ['radiation_station', 'pm25_lag1', 'wind_speed', 'no_sisaire', 'no2', 'pmc_lag2', 'temp_2m']
(1389, 262) {'radiation_station': 'W/m^2', 'pm25_lag1': 'kg/m^3', 'wind_speed': 'm/s', 'no_sisaire': 'kg/m^3', 'no2': 'kg/m^3', 'pmc_lag2': 'kg/m^3', 'temp_2m': 'K'} years: [np.int32(2019), np.int32(2020), np.int32(2021), np.int32(2022)]


In [3]:
from pysr import PySRRegressor
fronts = {}
for seed in SEEDS:
    sr = PySRRegressor(**PYSR_KW, random_state=seed)
    sr.fit(X, y, variable_names=features, X_units=X_units, y_units=Y_UNITS)
    f = sr.equations_[["complexity", "loss", "equation"]].copy(); f["seed"] = seed
    f.to_csv(DATA_DIR / f"{CITY}_{TARGET}_pysr_{TAG}_pareto_front_seed{seed}.csv", index=False)
    fronts[seed] = f
    print("seed", seed, "done; best:", sr.get_best()["equation"])

Compiling Julia backend...


/path/to/project/.venv/lib/python3.11/site-packages/pysr/sr.py:2385: UserWarning: Note: Setting `random_state` without also setting `deterministic=True` and `parallelism='serial'` will result in non-deterministic searches.
  warnings.warn(


┌ Warning: You are using dimensional constraints, but `dimensional_constraint_penalty` was not set. The default penalty of `1000.0` will be used.
└ @ SymbolicRegression ~/.julia/packages/SymbolicRegression/O7o63/src/Configure.jl:73
[ Info: Started!
[ Info: Final population:
[ Info: Results saved to:



Expressions evaluated per second: 3.120e+04
Progress: 44 / 200 total iterations (22.375%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           2.034e+02  0.000e+00  y[m⁻³ kg] = 55.867[?]
3           1.856e+02  4.597e-02  y[m⁻³ kg] = pm25_lag1[m⁻³ kg] + 38.225[?]
4           1.775e+02  4.445e-02  y[m⁻³ kg] = sqrt(radiation_station[m⁻² W]) * 4.754...
                                              8[?]
5           1.705e+02  4.026e-02  y[m⁻³ kg] = pm25_lag1[m⁻³ kg] + (radiation_station...
                                              [m⁻² W] * 0.27051[?])
6           1.570e+02  8.255e-02  y[m⁻³ kg] = pm25_lag1[m⁻³ kg] + sqrt(radiation_sta...
                                              tion[m⁻² W] * 10.843[?])
8           1.532e+02  1.203e-02  y[m⁻³ kg] = (pm25_lag1[m⁻³ kg] * 0.7116[?]) +

/path/to/project/.venv/lib/python3.11/site-packages/pysr/sr.py:2385: UserWarning: Note: Setting `random_state` without also setting `deterministic=True` and `parallelism='serial'` will result in non-deterministic searches.
  warnings.warn(
┌ Warning: You are using dimensional constraints, but `dimensional_constraint_penalty` was not set. The default penalty of `1000.0` will be used.
└ @ SymbolicRegression ~/.julia/packages/SymbolicRegression/O7o63/src/Configure.jl:73
[ Info: Started!
[ Info: Final population:
[ Info: Results saved to:



Expressions evaluated per second: 3.610e+04
Progress: 51 / 200 total iterations (25.833%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           2.034e+02  0.000e+00  y[m⁻³ kg] = 55.861[?]
3           1.856e+02  4.596e-02  y[m⁻³ kg] = pm25_lag1[m⁻³ kg] + 38.106[?]
4           1.775e+02  4.448e-02  y[m⁻³ kg] = sqrt(radiation_station[m⁻² W] * 22.608...
                                              [?])
5           1.710e+02  3.717e-02  y[m⁻³ kg] = (radiation_station[m⁻² W] * 0.26348[?]...
                                              ) + pm25_lag1[m⁻³ kg]
6           1.597e+02  6.851e-02  y[m⁻³ kg] = pm25_lag1[m⁻³ kg] + sqrt(radiation_sta...
                                              tion[m⁻² W] * 11.635[?])
7           1.581e+02  1.018e-02  y[m⁻³ kg] = (pm25_lag1[m⁻³ kg] + (radiation_s

/path/to/project/.venv/lib/python3.11/site-packages/pysr/sr.py:2385: UserWarning: Note: Setting `random_state` without also setting `deterministic=True` and `parallelism='serial'` will result in non-deterministic searches.
  warnings.warn(
┌ Warning: You are using dimensional constraints, but `dimensional_constraint_penalty` was not set. The default penalty of `1000.0` will be used.
└ @ SymbolicRegression ~/.julia/packages/SymbolicRegression/O7o63/src/Configure.jl:73
[ Info: Started!
[ Info: Final population:
[ Info: Results saved to:



Expressions evaluated per second: 3.550e+04
Progress: 50 / 200 total iterations (25.250%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           2.034e+02  0.000e+00  y[m⁻³ kg] = 55.849[?]
3           1.856e+02  4.596e-02  y[m⁻³ kg] = pm25_lag1[m⁻³ kg] + 38.243[?]
4           1.775e+02  4.446e-02  y[m⁻³ kg] = sqrt(radiation_station[m⁻² W]) / 0.210...
                                              27[?]
5           1.704e+02  4.055e-02  y[m⁻³ kg] = (radiation_station[m⁻² W] * 0.26826[?]...
                                              ) + pm25_lag1[m⁻³ kg]
6           1.587e+02  7.139e-02  y[m⁻³ kg] = sqrt(radiation_station[m⁻² W] / 0.1004...
                                              [?]) + pm25_lag1[m⁻³ kg]
7           1.581e+02  3.557e-03  y[m⁻³ kg] = ((radiation_station[m⁻² W] / 6.5

In [4]:
FLOAT = re.compile(r"(?<![A-Za-z_0-9.])\d+\.\d+(?:[eE][+-]?\d+)?")
def make_model(eq, feats):
    vals = []
    def sub(m): vals.append(float(m.group(0))); return f"c{len(vals)-1}"
    s = FLOAT.sub(sub, eq)
    syms = {f: sp.Symbol(f) for f in feats}; cs = [sp.Symbol(f"c{i}") for i in range(len(vals))]
    expr = sp.sympify(s, locals={**syms, **{str(c): c for c in cs}, "sqrt": sp.sqrt, "log": sp.log, "exp": sp.exp})
    used = [f for f in feats if syms[f] in expr.free_symbols]
    keep = [i for i, c in enumerate(cs) if c in expr.free_symbols]
    fn = sp.lambdify([syms[f] for f in used] + [cs[i] for i in keep], expr, "numpy")
    p0 = np.array([vals[i] for i in keep])
    return (lambda Xt, *p: fn(*Xt, *p)), used, p0

yy = data[TARGET].to_numpy(float); years = data["year"].to_numpy(); YRS = sorted(set(years))
def score(eq, feats):
    model, used, p0 = make_model(eq, feats)
    if not used:                                   # constant equation: leave-one-year-out mean
        oof = np.concatenate([np.full((years == yr).sum(), np.mean(yy[years != yr])) for yr in YRS]); yo = np.concatenate([yy[years == yr] for yr in YRS])
        per = [r2_score(yy[years == yr], np.full((years == yr).sum(), np.mean(yy[years != yr]))) for yr in YRS]
        return dict(r2_insample=0.0, r2_pooled=r2_score(yo, oof), r2_mean_years=float(np.mean(per)), vars="")
    Xt = tuple(data[f].to_numpy(float) for f in used)
    with np.errstate(all="ignore"), warnings.catch_warnings():
        warnings.simplefilter("ignore")
        try: popt, _ = curve_fit(model, Xt, yy, p0=p0, maxfev=20000)
        except Exception: popt = p0
        pred = np.full(len(yy), np.nan)
        for yr in YRS:
            te = years == yr
            try: pf, _ = curve_fit(model, tuple(a[~te] for a in Xt), yy[~te], p0=popt, maxfev=20000)
            except Exception: pf = popt
            pred[te] = model(tuple(a[te] for a in Xt), *pf)
        ok = np.all(np.isfinite(pred)); ins = model(Xt, *popt)
        per = [r2_score(yy[years == yr], pred[years == yr]) for yr in YRS] if ok else [np.nan]
        return dict(r2_insample=r2_score(yy, ins) if np.all(np.isfinite(ins)) else np.nan, r2_pooled=r2_score(yy, pred) if ok else np.nan,
                    r2_mean_years=float(np.mean(per)), vars=",".join(used))

rows = []
for seed, f in fronts.items():
    for _, r in f.iterrows():
        if r.complexity <= MAX_COMPLEXITY_REPORTED: rows.append(dict(seed=str(seed), complexity=int(r.complexity), equation=r.equation, **score(r.equation, features)))
ff = pd.read_csv(DATA_DIR / FREEFORM_FRONT).sort_values("complexity")
ff_feats = sorted({t for e in ff.equation for t in re.findall(r"[A-Za-z_][A-Za-z0-9_]*", e) if t in data.columns})
for _, r in ff.iterrows():
    if r.complexity <= MAX_COMPLEXITY_REPORTED:
        try: rows.append(dict(seed="free-form", complexity=int(r.complexity), equation=r.equation, **score(r.equation, ff_feats)))
        except Exception as e: print("free-form eq skipped", r.complexity, e)
scored = pd.DataFrame(rows); scored.to_csv(DATA_DIR / f"{CITY}_{TARGET}_pysr_{TAG}_scored.csv", index=False)
pd.set_option("display.width", 250, "display.max_colwidth", 110)
print("n days", len(data)); print(scored[["seed", "complexity", "r2_pooled", "r2_mean_years", "vars"]].round(3).to_string(index=False))

n days 1389
     seed  complexity  r2_pooled  r2_mean_years                                                  vars
        0           1     -0.004         -0.010                                                      
        0           3      0.077          0.056                                             pm25_lag1
        0           4      0.114          0.116                                     radiation_station
        0           5      0.126          0.110                           radiation_station,pm25_lag1
        0           6      0.208          0.193                           radiation_station,pm25_lag1
        0           8      0.287          0.277                radiation_station,pm25_lag1,wind_speed
        0           9      0.287          0.276                radiation_station,pm25_lag1,wind_speed
        0          11      0.301          0.291                radiation_station,pm25_lag1,wind_speed
        0          13      0.335          0.335           radiation_st